# Getting started with morph-spines

This notebook demonstrates the core API of the `morph-spines` library using a small
sample dataset included in the repository.

In [ ]:
from morph_spines import (
    SpineType,
    load_morphology_with_spines,
)

## Loading data

Load a morphology-with-spines file. The file contains 2 neurons, each with 4 spines.

In [ ]:
filepath = "./data/morphology_with_spines/sample_neurons_with_spines.h5"

# Load everything: morphology, soma, and spines (with meshes preloaded)
m = load_morphology_with_spines(filepath, morphology_name="neuron_0", load_meshes=True)

print(f"Spine count: {m.spines.spine_count}")
print(f"Spine table columns: {list(m.spines.spine_table.columns)}")

## Spine table

The spine table is a pandas DataFrame with one row per spine.

In [ ]:
m.spines.spine_table

## Spine skeletons

Spine skeletons are accessible as NeuroM Neurite objects.

In [ ]:
for i, neurite in enumerate(m.spines.spine_skeletons):
    print(f"Spine {i}: length={neurite.length:.3f}, points={len(neurite.points)}")

## Spine meshes

Spine meshes are returned as `trimesh.Trimesh` objects.

In [ ]:
# Full mesh for spine 0
mesh = m.spines.spine_mesh(0)
print(f"Spine 0 mesh: {len(mesh.vertices)} vertices, {len(mesh.faces)} faces")

# Centered (local coordinates) mesh
centered = m.spines.centered_spine_mesh(0)
print(f"Spine 0 centered mesh: {len(centered.vertices)} vertices, {len(centered.faces)} faces")

## Head/neck filtering

Spine meshes can be filtered to return only the head or neck region.
This is useful for analyzing spine morphology or rendering head and neck
with different colors.

In [ ]:
# Get only the neck
neck_mesh = m.spines.spine_mesh(0, include_head=False)
print(f"Neck: {len(neck_mesh.vertices)} vertices, {len(neck_mesh.faces)} faces")

# Get only the head
head_mesh = m.spines.spine_mesh(0, include_neck=False)
print(f"Head: {len(head_mesh.vertices)} vertices, {len(head_mesh.faces)} faces")

# Full mesh for comparison
full_mesh = m.spines.spine_mesh(0)
print(f"Full: {len(full_mesh.vertices)} vertices, {len(full_mesh.faces)} faces")

In [ ]:
# Visualize head (red) and neck (blue) for all spines
from trimesh import util as triutil
from trimesh.visual.color import ColorVisuals

HEAD_COLOR = [255, 100, 100, 255]
NECK_COLOR = [100, 100, 255, 255]

colored_parts = []
for spine_idx in range(m.spines.spine_count):
    head = m.spines.spine_mesh(spine_idx, include_neck=False)
    if len(head.faces) > 0:
        head.visual = ColorVisuals(mesh=head, face_colors=HEAD_COLOR)
        colored_parts.append(head)

    neck = m.spines.spine_mesh(spine_idx, include_head=False)
    if len(neck.faces) > 0:
        neck.visual = ColorVisuals(mesh=neck, face_colors=NECK_COLOR)
        colored_parts.append(neck)

head_neck_mesh = triutil.concatenate(colored_parts)
head_neck_mesh.show()

## Section-based queries

You can query spines by the morphology section they belong to.

In [ ]:
# Which sections have spines?
sections_with_spines = m.spines.spine_table["afferent_section_id"].unique()
print(f"Sections with spines: {sections_with_spines}")

# Get spine indices for a section
sec_id = sections_with_spines[0]
indices = m.spines.spine_indices_for_section(sec_id)
print(f"Section {sec_id} has spines at indices: {indices}")

# Get a compound mesh for all spines on a section
compound = m.spines.compound_spine_mesh_for_section(sec_id)
print(f"Compound mesh for section {sec_id}: {len(compound.faces)} faces")

## Spine type

If the spine table contains a `spine_type` column, you can access the
morphological classification. Otherwise, it returns `UNDEFINED`.

In [ ]:
# This sample data doesn't have spine_type in the table, so it returns UNDEFINED
print(f"Spine 0 type: {m.spines.spine_type(0)}")
print(f"Is undefined: {m.spines.spine_type(0) == SpineType.UNDEFINED}")

# Available spine types:
print(f"\nAll spine types: {[t.value for t in SpineType]}")

## Multiple neurons

The file contains multiple neurons. You can load each one by name.

In [ ]:
m1 = load_morphology_with_spines(filepath, morphology_name="neuron_1", load_meshes=True)
print(f"Neuron 1 spine count: {m1.spines.spine_count}")
print(f"Neuron 1 spine 0 mesh faces: {len(m1.spines.spine_mesh(0).faces)}")